#### Hierarchy:
Vectorized ops  >  map  >  apply  >  for-loops


### Vectorized Column Operations

In [84]:
import pandas as pd

df = pd.DataFrame({
    "user_id": [1, 2, 3, 4],
    "age": [17, 25, 40, 65],
    "city": ["Pune", "Mumbai", "Delhi", "Pune"],
    "salary": [20000, 50000, 80000, 30000]
})

#### Example: Increase salary by 10%

In [85]:
df["salary"] = df["salary"] * 1.10

#### Conditional Transformation

In [86]:
df["is_minor"] = (df["age"] < 18).astype("boolean")

In [87]:
df

,user_id,age,city,salary,is_minor
0,1,17,Pune,22000.0,True
1,2,25,Mumbai,55000.0,False
2,3,40,Delhi,88000.0,False
3,4,65,Pune,33000.0,False


### Creating Derived Columns

#### Age Group Example

In [88]:
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 18, 40, 60, 120],
    labels=["child", "adult", "middle_aged", "senior"]
)

df

,user_id,age,city,salary,is_minor,age_group
0,1,17,Pune,22000.0,True,child
1,2,25,Mumbai,55000.0,False,adult
2,3,40,Delhi,88000.0,False,adult
3,4,65,Pune,33000.0,False,senior


Used heavily in:
- ML features
- Analytics
- Business rules

#### Explanation:
`pd.cut` is a pandas function used to **bin continuous numerical data into discrete intervals.**

Here, it takes the numeric `age` values and assigns each one to a category.

`bins = [0, 18, 40, 60, 120]`    
These define the **interval boundaries:**
- 0-18
- 18-40
- 40-60
- 60-120

By default:
- Intervals are **right inclusive** -> `(0, 18]`, `(18, 40]`, etc.
- So `18` belongs to `child`, `40` to `adult`, `60` to `middle_aged`.

`labels = ["child", "adult", "middle_aged", "senior"]`
These are the **names assigned to each bin**, in order.

| Age range      | Label       |
| -------------- | ----------- |
| 0 < age ≤ 18   | child       |
| 18 < age ≤ 40  | adult       |
| 40 < age ≤ 60  | middle_aged |
| 60 < age ≤ 120 | senior      |


### `map()` - Element-wise Mapping
Use when:
- Simple value-to-value mapping
- Dictionary or function

Missing keys → `NaN`

In [89]:
city_code = {
    "Pune": "PNQ",
    "Mumbai": "BOM",
    "Delhi": "DEL"
}

df["city_code"] = df["city"].map(city_code)
df

,user_id,age,city,salary,is_minor,age_group,city_code
0,1,17,Pune,22000.0,True,child,PNQ
1,2,25,Mumbai,55000.0,False,adult,BOM
2,3,40,Delhi,88000.0,False,adult,DEL
3,4,65,Pune,33000.0,False,senior,PNQ


#### `map()` with function
- Clean
- Faster than `apply`

In [90]:
df["salary_k"] = df["salary"].map(lambda x: x / 1000)
df

,user_id,age,city,salary,is_minor,age_group,city_code,salary_k
0,1,17,Pune,22000.0,True,child,PNQ,22.0
1,2,25,Mumbai,55000.0,False,adult,BOM,55.0
2,3,40,Delhi,88000.0,False,adult,DEL,88.0
3,4,65,Pune,33000.0,False,senior,PNQ,33.0


### `apply()` - Row or Column-wise Logic
#### Row-wise apply (`axis=1`)

In [91]:
# Function receives one row of the DataFrame at a time
def label_user(row):
    if row["age"] < 18:
        return "minor"
    elif row["salary"] > 60000:
        return "high_earner"
    else:
        return "regular"
    
df["user_type"] = df.apply(label_user, axis=1)
df

,user_id,age,city,salary,is_minor,age_group,city_code,salary_k,user_type
0,1,17,Pune,22000.0,True,child,PNQ,22.0,minor
1,2,25,Mumbai,55000.0,False,adult,BOM,55.0,regular
2,3,40,Delhi,88000.0,False,adult,DEL,88.0,high_earner
3,4,65,Pune,33000.0,False,senior,PNQ,33.0,regular


#### Column-wise apply (`axis=0`)

In [92]:
df[["age", "salary"]].apply(lambda col: col.max() - col.min())

age          48.0
salary    66000.0
dtype: float64

### Conditional Updates with `loc`
`df.loc[condition, "tax_rate"]`
- `df.loc[...]` selects **rows that meet a condition**
- `"tax_rate"` is the column being **updated**
- If `"tax_rate"` doesn’t exist yet, pandas **creates it**

In [93]:
df.loc[df["salary"] > 60000, "tax_rate"] = 0.30
df.loc[df["salary"] <= 60000, "tax_rate"] = 0.20

### String Transformations (`.str` accessor)

In [94]:
df["city_upper"] = df["city"].str.upper()
df["city_len"] = df["city"].str.len()

- Works only on string dtype
- Vectorized and fast

In [95]:
df

,user_id,age,city,salary,is_minor,age_group,city_code,salary_k,user_type,tax_rate,city_upper,city_len
0,1,17,Pune,22000.0,True,child,PNQ,22.0,minor,0.2,PUNE,4
1,2,25,Mumbai,55000.0,False,adult,BOM,55.0,regular,0.2,MUMBAI,6
2,3,40,Delhi,88000.0,False,adult,DEL,88.0,high_earner,0.3,DELHI,5
3,4,65,Pune,33000.0,False,senior,PNQ,33.0,regular,0.2,PUNE,4


### Multiple Column Transformations
```python
df = (
    df
    .assign(...)
)
```

- The parentheses allow **method chaining**
- `assign`returns a **new DataFrame**
- The result is reassigned back to `df`
- The original `df` is **not modified in place**

In [96]:
df = (
    df.assign(
        salary_k = lambda x: x["salary"] / 1000,
        is_senior = lambda x: x["age"] >= 60
    )
)

df

,user_id,age,city,salary,is_minor,age_group,city_code,salary_k,user_type,tax_rate,city_upper,city_len,is_senior
0,1,17,Pune,22000.0,True,child,PNQ,22.0,minor,0.2,PUNE,4,False
1,2,25,Mumbai,55000.0,False,adult,BOM,55.0,regular,0.2,MUMBAI,6,False
2,3,40,Delhi,88000.0,False,adult,DEL,88.0,high_earner,0.3,DELHI,5,False
3,4,65,Pune,33000.0,False,senior,PNQ,33.0,regular,0.2,PUNE,4,True


### Summary
| Task                | Best Tool       |
| ------------------- | --------------- |
| Math on columns     | Vectorized ops  |
| Simple mapping      | `map()`         |
| Complex row logic   | `apply(axis=1)` |
| Conditional update  | `loc`           |
| String ops          | `.str`          |
| Multiple transforms | `assign()`      |


### Exercise 1
```python
df = pd.DataFrame({
    "user_id": [101, 102, 103, 104, 105],
    "age": [16, 25, 42, 67, 35],
    "city": ["Pune", "Mumbai", "Delhi", "Pune", "Mumbai"],
    "salary": [18000, 52000, 88000, 30000, 61000]
})
```

1. Create a column `salary_k` -> salary in thousands
2. Increase salary by **10%** for everyone (update column)

In [97]:
import pandas as pd

df = pd.DataFrame({
    "user_id": [101, 102, 103, 104, 105],
    "age": [16, 25, 42, 67, 35],
    "city": ["Pune", "Mumbai", "Delhi", "Pune", "Mumbai"],
    "salary": [18000, 52000, 88000, 30000, 61000]
})

In [98]:
df["salary_k"] = df["salary"] / 1000
df["salary"] = df["salary"] * 1.10
df

,user_id,age,city,salary,salary_k
0,101,16,Pune,19800.0,18.0
1,102,25,Mumbai,57200.0,52.0
2,103,42,Delhi,96800.0,88.0
3,104,67,Pune,33000.0,30.0
4,105,35,Mumbai,67100.0,61.0


### Exercise 2
1. Create is_minor → True if age < 18
2. Create is_senior → True if age >= 60

In [99]:
df["is_minor"] = (df["age"] < 18).astype("boolean")
df["is_senior"] = (df["age"] >= 60).astype("boolean")
df

,user_id,age,city,salary,salary_k,is_minor,is_senior
0,101,16,Pune,19800.0,18.0,True,False
1,102,25,Mumbai,57200.0,52.0,False,False
2,103,42,Delhi,96800.0,88.0,False,False
3,104,67,Pune,33000.0,30.0,False,True
4,105,35,Mumbai,67100.0,61.0,False,False


### Exercise 3
Create a column `age_group` with values:
| Age Range | Label           |
| --------- | --------------- |
| < 18      | `"child"`       |
| 18–40     | `"adult"`       |
| 41–60     | `"middle_aged"` |
| > 60      | `"senior"`      |


In [100]:
df["age_group"] = pd.cut(
    df["age"],
    bins= [0, 18, 40, 60, 100],
    labels= ["child", "adult", "middle_aged", "senior"]
)

df

,user_id,age,city,salary,salary_k,is_minor,is_senior,age_group
0,101,16,Pune,19800.0,18.0,True,False,child
1,102,25,Mumbai,57200.0,52.0,False,False,adult
2,103,42,Delhi,96800.0,88.0,False,False,middle_aged
3,104,67,Pune,33000.0,30.0,False,True,senior
4,105,35,Mumbai,67100.0,61.0,False,False,adult


### Exercise 4
1. Map cities to codes:
    - Pune -> `"PNQ"`
    - Mumbai -> `"BOM"`
    - Delhi -> `"DEL"`
    
2.Store in `city_code`

In [101]:
city_code = {
    "Pune": "PNQ",
    "Mumbai": "BOM",
    "Delhi": "DEL"
}

df["city_code"] = df["city"].map(city_code)
df


,user_id,age,city,salary,salary_k,is_minor,is_senior,age_group,city_code
0,101,16,Pune,19800.0,18.0,True,False,child,PNQ
1,102,25,Mumbai,57200.0,52.0,False,False,adult,BOM
2,103,42,Delhi,96800.0,88.0,False,False,middle_aged,DEL
3,104,67,Pune,33000.0,30.0,False,True,senior,PNQ
4,105,35,Mumbai,67100.0,61.0,False,False,adult,BOM


### Exercise 5
1. Create tax_rate:
    - salary > 60000 -> `0.30`
    - salary <= 60000 -> `0.20`

In [102]:
df.loc[df["salary"] > 60000, "tax_rate"] = 0.30
df.loc[df["salary"] <= 60000, "tax_rate"] = 0.20
df

,user_id,age,city,salary,salary_k,is_minor,is_senior,age_group,city_code,tax_rate
0,101,16,Pune,19800.0,18.0,True,False,child,PNQ,0.2
1,102,25,Mumbai,57200.0,52.0,False,False,adult,BOM,0.2
2,103,42,Delhi,96800.0,88.0,False,False,middle_aged,DEL,0.3
3,104,67,Pune,33000.0,30.0,False,True,senior,PNQ,0.2
4,105,35,Mumbai,67100.0,61.0,False,False,adult,BOM,0.3


### Exercise 6
Create `user_segment`:
- `"student"` -> age < 18
- `"high_earner"` -> salary > 80000
- `"regular"` -> otherwise

In [103]:
def assign_label(row):
    if row["age"] < 18:
        return "student"
    elif row["salary"] > 80000:
        return "high_earner"
    else:
        return "regular"
    
df["user_segment"] = df.apply(assign_label, axis=1)
df

,user_id,age,city,salary,salary_k,is_minor,is_senior,age_group,city_code,tax_rate,user_segment
0,101,16,Pune,19800.0,18.0,True,False,child,PNQ,0.2,student
1,102,25,Mumbai,57200.0,52.0,False,False,adult,BOM,0.2,regular
2,103,42,Delhi,96800.0,88.0,False,False,middle_aged,DEL,0.3,high_earner
3,104,67,Pune,33000.0,30.0,False,True,senior,PNQ,0.2,regular
4,105,35,Mumbai,67100.0,61.0,False,False,adult,BOM,0.3,regular


### Cleaner, Production-Style Rewrite (Exercise 1-5)

In [105]:
df = (
    df
    .assign(
        salary_k=lambda x: x["salary"] / 1000,
        salary=lambda x: x["salary"] * 1.10,
        is_minor=lambda x: x["age"] < 18,
        is_senior=lambda x: x["age"] >= 60,
        age_group=lambda x: pd.cut(
            x["age"],
            bins=[0, 18, 40, 60, 100],
            labels=["child", "adult", "middle_aged", "senior"]
        ),
        city_code=lambda x: x["city"].map({
            "Pune": "PNQ",
            "Mumbai": "BOM",
            "Delhi": "DEL"
        })
    )
)

df["tax_rate"] = 0.20
df.loc[df["salary"] > 60000, "tax_rate"] = 0.30
df

,user_id,age,city,salary,salary_k,is_minor,is_senior,age_group,city_code,tax_rate,user_segment
0,101,16,Pune,23958.0,21.78,True,False,child,PNQ,0.2,student
1,102,25,Mumbai,69212.0,62.92,False,False,adult,BOM,0.3,regular
2,103,42,Delhi,117128.0,106.48,False,False,middle_aged,DEL,0.3,high_earner
3,104,67,Pune,39930.0,36.30,False,True,senior,PNQ,0.2,regular
4,105,35,Mumbai,81191.0,73.81,False,False,adult,BOM,0.3,regular
